In [ ]:
import numpy as np
from scipy.fft import dct, dst

In [ ]:
COS_PI_BY_4 = np.sqrt(0.5)

def discrete_sine_and_cosine_type_2(data: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
  size = data.size
  if size == 1:
    return 2*data, 2*data

  assert size % 2 == 0

  data_even = data[::2]
  data_odd  = data[1::2]

  cos_even, sin_even = discrete_sine_and_cosine_type_2(data_even)
  cos_odd,  sin_odd  = discrete_sine_and_cosine_type_2(data_odd)

  cos_data = np.zeros_like(data, dtype=float)
  sin_data = np.zeros_like(data, dtype=float)

  cos_data[0]             = cos_even[0] + cos_odd[0]
  sin_data[size - 1]      = cos_even[0] - cos_odd[0]
  cos_data[size // 2]     = COS_PI_BY_4 * (sin_even[size // 2 - 1] - sin_odd[size // 2 - 1])
  sin_data[size // 2 - 1] = COS_PI_BY_4 * (sin_even[size // 2 - 1] + sin_odd[size // 2 - 1])
  for k in range(1, size // 2):
    twiddle_sin = np.sin(np.pi * k / (size * 2))
    twiddle_cos = np.cos(np.pi * k / (size * 2))
    cos_data[k]     =   (cos_even[k]   + cos_odd[k])   * twiddle_cos \
                      + (sin_even[k-1] - sin_odd[k-1]) * twiddle_sin
    sin_data[k - 1] =   (sin_even[k-1] + sin_odd[k-1]) * twiddle_cos \
                      - (cos_even[k]   - cos_odd[k])   * twiddle_sin

    # odd/even element transforms have not explicit values for k >= size//2
    # but they fulfill symetries across k = size//2:
    #   cos_trf[N + k] = - cos_trf[N - k]
    #   sin_trf[N + k] = sin[N - k]
    # We get the values by "reflecting" across size//2
    k_refl = size // 2 - k
    # cos(pi (N//2 + k) / (2 N)) = cos(pi/4 + (pi k)/(2N)) = cos(pi / 4) cos((pi k) / (2N)) - sin(pi / 4) sin((pi k) / (2N))
    # similarly for sin; and use sin(pi/4) = cos(pi/4) = sqrt(1/2)
    twiddle_cos_refl = COS_PI_BY_4 * (twiddle_cos - twiddle_sin)
    twiddle_sin_refl = COS_PI_BY_4 * (twiddle_cos + twiddle_sin)

    cos_data[size//2 + k]     = - (cos_even[k_refl]     + cos_odd[k_refl])     * twiddle_cos_refl \
                                + (sin_even[k_refl - 1] - sin_odd[k_refl - 1]) * twiddle_sin_refl
    sin_data[size//2 + k - 1] =   (sin_even[k_refl - 1] + sin_odd[k_refl - 1]) * twiddle_cos_refl \
                                + (cos_even[k_refl]     - cos_odd[k_refl])     * twiddle_sin_refl

  return cos_data, sin_data


def discrete_sine_and_cosine_type_2_unrolled(data: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
  size = data.size
  if size == 1:
    return 2*data, 2*data

  log2_size = 1
  while (size >> log2_size) != 1:
    log2_size += 1
  assert size == (1 << log2_size)

  cos_prev = data.astype(float)
  sin_prev = data.astype(float)
  cos_curr = np.zeros_like(data, float)
  sin_curr = np.zeros_like(data, float)
  for stage in range(log2_size):
    stride = (size >> (stage + 1)) # first stage is 0 -> stride is size // 2
    block_size = 1 << (stage + 1)  # first stage is 0 -> block size is 2
    for block_idx in range(stride):
      # Special handling of k == 0 due to k offset between sin/cos
      cos_even_k0    = cos_prev[block_idx]
      cos_odd_k0     = cos_prev[block_idx + stride]
      sin_even_khalf = sin_prev[block_idx + 2 * (block_size // 2 - 1) * stride]
      sin_odd_khalf  = sin_prev[block_idx + (2 * (block_size // 2 - 1) + 1) * stride]

      cos_curr[block_idx]                             = cos_even_k0 + cos_odd_k0
      sin_curr[block_idx + (block_size - 1) * stride] = cos_even_k0 - cos_odd_k0
      cos_curr[block_idx + (block_size // 2) * stride]     = (sin_even_khalf - sin_odd_khalf) * COS_PI_BY_4
      sin_curr[block_idx + (block_size // 2 - 1) * stride] = (sin_even_khalf + sin_odd_khalf) * COS_PI_BY_4

    for k in range(1, block_size // 2):
      for block_idx in range(stride):
        twiddle_sin = np.sin(np.pi * k / (block_size * 2))
        twiddle_cos = np.cos(np.pi * k / (block_size * 2))

        cos_even = cos_prev[block_idx + (2 * k) * stride]
        cos_odd  = cos_prev[block_idx + (2 * k + 1) * stride]
        sin_even = sin_prev[block_idx + (2 * (k - 1)) * stride]
        sin_odd  = sin_prev[block_idx + (2 * (k - 1) + 1) * stride]

        cos_sum  = cos_even + cos_odd
        cos_diff = cos_even - cos_odd
        sin_sum  = sin_even + sin_odd
        sin_diff = sin_even - sin_odd

        cos_curr[block_idx + k * stride]       = cos_sum * twiddle_cos + sin_diff * twiddle_sin
        sin_curr[block_idx + (k - 1) * stride] = sin_sum * twiddle_cos - cos_diff * twiddle_sin

        # odd/even element transforms don't have explicit values for k >= size//2
        # but they fulfill symetries across k = size//2:
        #   cos_trf[N + k] = - cos_trf[N - k]
        #   sin_trf[N + k] = sin[N - k]
        # We get the values by "reflecting" across size//2: k_refl = N // 2 - k
        # which gives us the result for output index N//2 + k_refl == N - k
        # Also, the twiddle factors at k_refl are sin/cos(pi * (N//2 + k_refl) / (2 * N)) = sin/cos(pi * (N - k) / (2 * N)) = sin/cos(pi/2 - (pi k) (2 N))
        # => twiddle_sin(k_refl) = twiddle_cos(k),
        #    twiddle_cos(k_refl) = twiddle_sin(k)

        cos_curr[block_idx + (block_size - k) * stride]     = - cos_sum * twiddle_sin + sin_diff * twiddle_cos
        sin_curr[block_idx + (block_size - k - 1) * stride] =   sin_sum * twiddle_sin + cos_diff * twiddle_cos

    cos_prev[:] = cos_curr
    sin_prev[:] = sin_curr

  return 2 * cos_curr, 2 * sin_curr


def discrete_cosine_type_2_unrolled(data: np.ndarray):
  data = data.astype(float)
  size = data.size

  cos_even, sin_even = discrete_sine_and_cosine_type_2_unrolled(data[::2])
  cos_odd,  sin_odd  = discrete_sine_and_cosine_type_2_unrolled(data[1::2])

  cos_data = np.zeros_like(data)
  cos_data[0]         = cos_even[0] + cos_odd[0]
  cos_data[size // 2] = COS_PI_BY_4 * (sin_even[size // 2 - 1] - sin_odd[size // 2 - 1])
  for k in range(1, size // 2):
    twiddle_sin = np.sin(np.pi * k / (size * 2))
    twiddle_cos = np.cos(np.pi * k / (size * 2))

    cos_sum  = cos_even[k]     + cos_odd[k]
    sin_diff = sin_even[k - 1] - sin_odd[k - 1]

    cos_data[k]        =   cos_sum * twiddle_cos + sin_diff * twiddle_sin
    cos_data[size - k] = - cos_sum * twiddle_sin + sin_diff * twiddle_cos

  return cos_data


def discrete_sine_type_2_unrolled(data: np.ndarray):
  data = data.astype(float)
  size = data.size

  cos_even, sin_even = discrete_sine_and_cosine_type_2_unrolled(data[::2])
  cos_odd,  sin_odd  = discrete_sine_and_cosine_type_2_unrolled(data[1::2])

  sin_data = np.zeros_like(data)
  sin_data[size - 1]      = cos_even[0] - cos_odd[0]
  sin_data[size // 2 - 1] = COS_PI_BY_4 * (sin_even[size // 2 - 1] + sin_odd[size // 2 - 1])
  for k in range(1, size // 2):
    twiddle_sin = np.sin(np.pi * k / (size * 2))
    twiddle_cos = np.cos(np.pi * k / (size * 2))

    sin_sum  = sin_even[k - 1] + sin_odd[k - 1]
    cos_diff = cos_even[k]     - cos_odd[k]

    sin_data[k - 1]        = sin_sum * twiddle_cos - cos_diff * twiddle_sin
    sin_data[size - k - 1] = sin_sum * twiddle_sin + cos_diff * twiddle_cos

  return sin_data



In [ ]:
arr = np.random.rand(1 << 16)

c, s = discrete_sine_and_cosine_type_2_unrolled(arr)
cr, sr = dct(arr), dst(arr)

np.allclose(c, cr) and np.allclose(s, sr)

In [ ]:
arr = np.random.rand(1 << 16)

c = discrete_cosine_type_2_unrolled(arr)
s = discrete_sine_type_2_unrolled(arr)
cr = dct(arr)
sr = dst(arr)

np.allclose(c, cr) and np.allclose(s, sr)

In [ ]:
def _discrete_sine_and_cosine_type_3_pre(data: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
  size = data.size
  if size == 1:
    return 2 * data, np.zeros_like(data)

  assert size % 2 == 0

  data_even = data[::2]
  data_odd  = data[1::2]

  cos_even, sin_even = _discrete_sine_and_cosine_type_3_pre(data_even)
  cos_odd,  sin_odd  = _discrete_sine_and_cosine_type_3_pre(data_odd)

  cos_data = np.zeros_like(data, dtype=float)
  sin_data = np.zeros_like(data, dtype=float)

  for k in range(size // 2):
    twiddle_sin = np.sin(np.pi * (k + 0.5) / size)
    twiddle_cos = np.cos(np.pi * (k + 0.5) / size)
    cos_data[k] = cos_even[k] + cos_odd[k] * twiddle_cos \
                              - sin_odd[k] * twiddle_sin
    sin_data[k] = sin_even[k] + sin_odd[k] * twiddle_cos \
                              + cos_odd[k] * twiddle_sin

    # odd/even element transforms have not explicit values for k >= size//2
    # but they fulfill symetries across k = size//2 - 1/2:
    #   cos_trf[N + k] =   cos_trf[N - k - 1]
    #   sin_trf[N + k] = - sin[N - k - 1]
    # We get the values by "reflecting" across size//2
    k_refl = size // 2 - k - 1

    cos_data[size//2 + k] =   cos_even[k_refl] - cos_odd[k_refl] * twiddle_sin \
                                               + sin_odd[k_refl] * twiddle_cos
    sin_data[size//2 + k] = - sin_even[k_refl] + sin_odd[k_refl] * twiddle_sin \
                                               + cos_odd[k_refl] * twiddle_cos

  return cos_data, sin_data


def _discrete_cosine_type_3_pre(data: np.ndarray):
  size = data.size
  if size == 1:
    return 2 * data

  assert size % 2 == 0

  data_even = data[::2]
  data_odd  = data[1::2]

  cos_even = _discrete_cosine_type_3_pre(data_even)
  cos_odd, sin_odd = _discrete_sine_and_cosine_type_3_pre(data_odd)

  cos_data = np.zeros_like(data, dtype=float)

  for k in range(size // 2):
    twiddle_sin = np.sin(np.pi * (k + 0.5) / size)
    twiddle_cos = np.cos(np.pi * (k + 0.5) / size)
    cos_data[k] = cos_even[k] + cos_odd[k] * twiddle_cos \
                              - sin_odd[k] * twiddle_sin

    # odd/even element transforms have not explicit values for k >= size//2
    # but they fulfill symetries across k = size//2 - 1/2:
    #   cos_trf[N + k] =   cos_trf[N - k - 1]
    #   sin_trf[N + k] = - sin[N - k - 1]
    # We get the values by "reflecting" across size//2
    k_refl = size // 2 - k - 1

    cos_data[size//2 + k] = cos_even[k_refl] - cos_odd[k_refl] * twiddle_sin \
                                             + sin_odd[k_refl] * twiddle_cos

  return cos_data


def discrete_cosine_type_3(data: np.ndarray):
  size = data.size
  if size == 1:
    return 2 * data
  data = data.astype(float)
  data[0] *= 0.5

  return _discrete_cosine_type_3_pre(data)


def discrete_cosine_type_3_unrolled(data: np.ndarray):
  size = data.size
  if size == 1:
    return 2.0*data

  log2_size = 1
  while (size >> log2_size) != 1:
    log2_size += 1
  assert size == (1 << log2_size)

  cos_prev = data.astype(float)
  cos_prev[1:] *= 2.0
  sin_prev = np.zeros_like(data, float)
  cos_curr = np.zeros_like(data, float)
  sin_curr = np.zeros_like(data, float)

  for stage in range(log2_size):
    stride = (size >> (stage + 1)) # first stage is 0 -> stride is size // 2
    block_size = 1 << (stage + 1)  # first stage is 0 -> block size is 2
    for k in range(block_size // 2):
      twiddle_sin = np.sin(np.pi * (k + 0.5) / block_size)
      twiddle_cos = np.cos(np.pi * (k + 0.5) / block_size)

      # Block 0 never needs the even sine transform
      cos_even = cos_prev[(2 * k) * stride]
      cos_odd  = cos_prev[(2 * k + 1) * stride]
      sin_odd  = sin_prev[(2 * k + 1) * stride]

      cos_odd_term = cos_odd * twiddle_cos - sin_odd * twiddle_sin

      cos_curr[k * stride]                    = cos_even + cos_odd_term
      cos_curr[(block_size - k - 1) * stride] = cos_even - cos_odd_term

      for block_idx in range(1, stride):
        cos_even = cos_prev[block_idx + (2 * k) * stride]
        cos_odd  = cos_prev[block_idx + (2 * k + 1) * stride]
        sin_even = sin_prev[block_idx + (2 * k) * stride]
        sin_odd  = sin_prev[block_idx + (2 * k + 1) * stride]

        cos_odd_term = cos_odd * twiddle_cos - sin_odd * twiddle_sin
        sin_odd_term = sin_odd * twiddle_cos + cos_odd * twiddle_sin

        cos_curr[block_idx + k * stride] = cos_even + cos_odd_term
        sin_curr[block_idx + k * stride] = sin_even + sin_odd_term

        # odd/even element transforms have not explicit values for k >= size//2
        # but they fulfill symetries across k = size//2 - 1/2:
        #   cos_trf[N + k] =   cos_trf[N - k - 1]
        #   sin_trf[N + k] = - sin[N - k - 1]
        # We get the values by "reflecting" across size//2
        cos_curr[block_idx + (block_size - k - 1) * stride] =   cos_even - cos_odd_term
        sin_curr[block_idx + (block_size - k - 1) * stride] = - sin_even + sin_odd_term

    cos_prev[:] = cos_curr
    sin_prev[:] = sin_curr

  return cos_curr


def discrete_sine_type_3_unrolled(data: np.ndarray):
  size = data.size
  if size == 1:
    return data

  log2_size = 1
  while (size >> log2_size) != 1:
    log2_size += 1
  assert size == (1 << log2_size)

  sin_prev = data.astype(float)
  sin_prev[:-1] *= 2.0
  cos_prev = np.zeros_like(data, float)
  sin_curr = np.zeros_like(data, float)
  cos_curr = np.zeros_like(data, float)

  for stage in range(log2_size):
    stride = (size >> (stage + 1)) # first stage is 0 -> stride is size // 2
    block_size = 1 << (stage + 1)  # first stage is 0 -> block size is 2
    for k in range(block_size // 2):
      twiddle_sin = np.sin(np.pi * (k + 0.5) / block_size)
      twiddle_cos = np.cos(np.pi * (k + 0.5) / block_size)

      for block_idx in range(stride - 1):
        sin_even = sin_prev[block_idx + (2 * k) * stride]
        sin_odd  = sin_prev[block_idx + (2 * k + 1) * stride]
        cos_even = cos_prev[block_idx + (2 * k) * stride]
        cos_odd  = cos_prev[block_idx + (2 * k + 1) * stride]

        sin_even_term = sin_even * twiddle_cos - cos_even * twiddle_sin
        cos_even_term = cos_even * twiddle_cos + sin_even * twiddle_sin

        sin_curr[block_idx + k * stride] = sin_odd + sin_even_term
        cos_curr[block_idx + k * stride] = cos_odd + cos_even_term

        # odd/even element transforms have not explicit values for k >= size//2
        # but they fulfill symetries across k = size//2 - 1/2:
        #   cos_trf[N + k] =   cos_trf[N - k - 1]
        #   sin_trf[N + k] = - sin[N - k - 1]
        # We get the values by "reflecting" across size//2
        sin_curr[block_idx + (block_size - k - 1) * stride] = - sin_odd + sin_even_term
        cos_curr[block_idx + (block_size - k - 1) * stride] =   cos_odd - cos_even_term

      # Block stride-1 never needs the odd cosine transform
      block_idx = stride - 1
      sin_even = sin_prev[block_idx + (2 * k) * stride]
      sin_odd  = sin_prev[block_idx + (2 * k + 1) * stride]
      cos_even = cos_prev[block_idx + (2 * k) * stride]

      sin_even_term = sin_even * twiddle_cos - cos_even * twiddle_sin

      sin_curr[block_idx + k * stride]                    =   sin_odd + sin_even_term
      sin_curr[block_idx + (block_size - k - 1) * stride] = - sin_odd + sin_even_term

    cos_prev[:] = cos_curr
    sin_prev[:] = sin_curr

  return sin_curr

In [ ]:
arr = np.random.rand(1 << 16)

d = discrete_cosine_type_3_unrolled(arr)
dr = dct(arr, type=3)

np.allclose(d, dr)

In [ ]:
arr = np.random.rand(1 << 16)

s = discrete_sine_type_3_unrolled(arr)
sr = dst(arr, type=3)

np.allclose(s, sr)